# Data Wrangling #

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import re

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [2]:
food = pd.read_sql_query("SELECT * FROM food", conn)
food_nutrient = pd.read_sql_query("SELECT * FROM food_nutrient", conn)
nutrient = pd.read_sql_query("SELECT * FROM nutrient", conn)
wal_price = pd.read_sql_query("SELECT * FROM walmart_price", conn)
wf_price = pd.read_sql_query("SELECT * FROM wholefoods_price", conn)

In [3]:
def normalize_text(str):
    if pd.isna(str):
        return ""
    str = str.lower()
    str = re.sub(r'[^a-z0-9\s]', ' ', str)
    str = re.sub(r'\s+', ' ', str).strip()
    return str

food["clean_desc"] = food["description"].apply(normalize_text)
wal_price["clean_name"] = wal_price["product_name"].apply(normalize_text)
wf_price["clean_name"] = wf_price["product_name"].apply(normalize_text)

food["clean_brand_owner"] = food["brand_owner"].apply(normalize_text)
food["clean_brand"] = food["brand_name"].apply(normalize_text)
food["clean_subbrand"] = food["subbrand_name"].apply(normalize_text)

wal_price["clean_brand"] = wal_price["brand"].apply(normalize_text)
wf_price["clean_brand"] = wf_price["brand"].apply(normalize_text)

In [4]:
food_brand_owners = food["clean_brand_owner"].dropna().unique().tolist()
food_brands = food["clean_brand"].dropna().unique().tolist()
food_subbrands = food["clean_subbrand"].dropna().unique().tolist()

wal_brands = wal_price["clean_brand"].dropna().unique().tolist()
wf_brands = wf_price["clean_brand"].dropna().unique().tolist()

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pathlib

model: SentenceTransformer = SentenceTransformer('all-MiniLM-L6-v2')

def load_embedding(path: pathlib.Path):
    if path.exists():
        return np.load(path)
    else:
        return None

def save_embedding(embedding, path: pathlib.Path):
    np.save(path, embedding)

wf_brand_emb = load_embedding(pathlib.Path('./data/wf_brand_emb.npy'))
if wf_brand_emb is None:
    print("Embedding could not be loaded")
    wf_brand_emb = model.encode(wf_brands, show_progress_bar=True)
    save_embedding(wf_brand_emb, pathlib.Path('./data/wf_brand_emb'))

wal_brand_emb = load_embedding(pathlib.Path('./data/wal_brand_emb.npy'))
if wal_brand_emb is None:
    print("Embedding could not be loaded")
    wal_brand_emb = model.encode(wal_brands, show_progress_bar=True)
    save_embedding(wal_brand_emb, pathlib.Path('./data/wal_brand_emb'))

food_brand_owner_emb = load_embedding(pathlib.Path('./data/food_brand_owner_emb.npy'))
if food_brand_owner_emb is None:
    print("Embedding could not be loaded")
    food_brand_owner_emb = model.encode(food_brand_owners, show_progress_bar=True)
    save_embedding(food_brand_owner_emb, pathlib.Path('./data/food_brand_owner_emb'))

food_brand_emb = load_embedding(pathlib.Path('./data/food_brand_emb.npy'))
if food_brand_emb is None:
    print("Embedding could not be loaded")
    food_brand_emb = model.encode(food_brands, show_progress_bar=True)
    save_embedding(food_brand_emb, pathlib.Path('./data/food_brand_emb'))

food_subbrand_emb = load_embedding(pathlib.Path('./data/food_subbrand_emb.npy'))
if food_subbrand_emb is None:
    print("Embedding could not be loaded")
    food_subbrand_emb = model.encode(food_subbrands, show_progress_bar=True)
    save_embedding(food_subbrand_emb, pathlib.Path('./data/food_subbrand_emb'))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from sentence_transformers import util
import numpy as np

SIMILARITY_THRESH = 0.95

wf_sim_matrix = util.cos_sim(wf_brand_emb, food_brand_emb)
wf_s_sim_matrix = util.cos_sim(wf_brand_emb, food_subbrand_emb)
wal_sim_matrix = util.cos_sim(wal_brand_emb, food_brand_emb)
wal_s_sim_matrix = util.cos_sim(wal_brand_emb, food_subbrand_emb)

wf_to_food = {}
for i, wf_brand in enumerate(wf_brands):
    best_idx = np.argmax(wf_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wf_sim_matrix[i][best_idx].item()

    if best_score > SIMILARITY_THRESH:
        wf_to_food[wf_brand] = (best_match, best_score)
    else:
        sub_best_idx = np.argmax(wf_s_sim_matrix[i]).item()
        sub_best_match = food_brands[sub_best_idx]
        sub_best_score = wf_s_sim_matrix[i][sub_best_idx].item()

        if sub_best_score > best_score:
            wf_to_food[wf_brand] = (sub_best_match, sub_best_score)
        else:
            wf_to_food[wf_brand] = (best_match, best_score)

wal_to_food = {}
for i, wal_brand in enumerate(wal_brands):
    best_idx = np.argmax(wal_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wal_sim_matrix[i][best_idx].item()
    
    if best_score > SIMILARITY_THRESH:
        wal_to_food[wal_brand] = (best_match, best_score)
    else:
        sub_best_idx = np.argmax(wal_s_sim_matrix[i]).item()
        sub_best_match = food_brands[sub_best_idx]
        sub_best_score = wal_s_sim_matrix[i][sub_best_idx].item()

        if sub_best_score > best_score:
            wal_to_food[wal_brand] = (sub_best_match, sub_best_score)
        else:
            wal_to_food[wal_brand] = (best_match, best_score)

wf_price['udsa_brand_match'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[0])
wf_price['udsa_brand_score'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[1])
wal_price['udsa_brand_match'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[0])
wal_price['udsa_brand_score'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[1])

In [7]:
food_desc_emb_path = pathlib.Path('./data/food-desc-emb.npy')
food_desc_emb = load_embedding(food_desc_emb_path)
if food_desc_emb is None:
    food_desc_emb = model.encode(food.clean_desc, show_progress_bar=True)
    save_embedding(food_desc_emb, food_desc_emb_path)

In [8]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

In [9]:
import faiss

MATCHING_SIMILARITY_THRESH = 0.90

print(f"{len(wf_brands)} total brands to match")

wf_emb = load_embedding(pathlib.Path('./data/wf_emb.npy'))
if wf_emb is None:
    print("Embedding could not be loaded")
    wf_emb = model.encode(wf_price.clean_name, show_progress_bar=True)
    save_embedding(wf_emb, pathlib.Path('./data/wf_emb'))

index = faiss.IndexFlatIP(food_desc_emb.shape[1])
index.add(normalize(food_desc_emb))

wf_item_to_food = {}
for i, brand in enumerate(wf_brands):
    mask = wf_price.clean_brand.eq(brand)

    branded_items = wf_price[mask]
    if branded_items.empty:
        continue

    print(f"Brand {i}: Matching {len(branded_items)} items")

    wf_item_emb = wf_emb[mask]
    wf_item_emb = normalize(wf_item_emb)

    scores, indices = index.search(wf_item_emb, 1)

    for i, (_, item) in enumerate(branded_items.iterrows()):
        best_idx = indices[i][0]
        best_score = scores[i][0]
        best_match = food.iloc[best_idx]

        if best_score > MATCHING_SIMILARITY_THRESH:
            wf_item_to_food[item.clean_name] = (best_match, best_score)

wf_price['usda_item_match'] = wf_price['clean_name'].map(lambda b: wf_item_to_food.get(b, (None, 0))[0])
wf_price['usda_item_score'] = wf_price['clean_name'].map(lambda b: wf_item_to_food.get(b, (None, 0))[1])

290 total brands to match
Brand 0: Matching 2 items
Brand 1: Matching 505 items
Brand 2: Matching 2 items
Brand 3: Matching 6 items
Brand 4: Matching 2 items
Brand 5: Matching 2 items
Brand 6: Matching 2 items
Brand 7: Matching 4 items
Brand 8: Matching 3 items
Brand 9: Matching 6 items
Brand 10: Matching 1 items
Brand 11: Matching 9 items
Brand 12: Matching 2 items
Brand 13: Matching 6 items
Brand 14: Matching 9 items
Brand 15: Matching 7 items
Brand 16: Matching 5 items
Brand 17: Matching 1 items
Brand 18: Matching 6 items
Brand 19: Matching 1 items
Brand 20: Matching 69 items
Brand 21: Matching 3 items
Brand 22: Matching 2 items
Brand 23: Matching 1 items
Brand 24: Matching 1 items
Brand 25: Matching 14 items
Brand 26: Matching 11 items
Brand 27: Matching 1 items
Brand 28: Matching 2 items
Brand 29: Matching 2 items
Brand 30: Matching 6 items
Brand 31: Matching 1 items
Brand 32: Matching 3 items
Brand 33: Matching 1 items
Brand 34: Matching 3 items
Brand 35: Matching 4 items
Brand 3

In [11]:
print(f"{len(wal_brands)} total brands to match")

wal_emb = load_embedding(pathlib.Path('./data/wal_emb.npy'))
if wal_emb is None:
    print("Embedding could not be loaded")
    wal_emb = model.encode(wal_price.clean_name, show_progress_bar=True)
    save_embedding(wal_emb, pathlib.Path('./data/wal_emb'))

wal_item_to_food = {}
for i, brand in enumerate(wal_brands):
    mask = wal_price.clean_brand.eq(brand)

    branded_items = wal_price[mask]
    if branded_items.empty:
        continue

    print(f"Brand {i}: Matching {len(branded_items)} items")

    wal_item_emb = wal_emb[mask]
    wal_item_emb = normalize(wal_item_emb)

    scores, indices = index.search(wal_item_emb, 1)

    for i, (_, item) in enumerate(branded_items.iterrows()):
        best_idx = indices[i][0]
        best_score = scores[i][0]
        best_match = food.iloc[best_idx]

        if best_score > MATCHING_SIMILARITY_THRESH:
            wal_item_to_food[item.clean_name] = (best_match, best_score)

wal_price['usda_item_match'] = wal_price['clean_name'].map(lambda b: wal_item_to_food.get(b, (None, 0))[0])
wal_price['usda_item_score'] = wal_price['clean_name'].map(lambda b: wal_item_to_food.get(b, (None, 0))[1])

4296 total brands to match
Brand 0: Matching 15772 items
Brand 1: Matching 133 items
Brand 2: Matching 137 items
Brand 3: Matching 415 items
Brand 4: Matching 17815 items
Brand 5: Matching 232 items
Brand 6: Matching 40 items
Brand 7: Matching 80387 items
Brand 8: Matching 41 items
Brand 9: Matching 99 items
Brand 10: Matching 1267 items
Brand 11: Matching 707 items
Brand 12: Matching 32 items
Brand 13: Matching 17 items
Brand 14: Matching 484 items
Brand 15: Matching 188 items
Brand 16: Matching 473 items
Brand 17: Matching 776 items
Brand 18: Matching 140 items
Brand 19: Matching 58 items
Brand 20: Matching 52 items
Brand 21: Matching 989 items
Brand 22: Matching 87 items
Brand 23: Matching 1341 items
Brand 24: Matching 80 items
Brand 25: Matching 16 items
Brand 26: Matching 105 items
Brand 27: Matching 4 items
Brand 28: Matching 3 items
Brand 29: Matching 595 items
Brand 30: Matching 519 items
Brand 31: Matching 1671 items
Brand 32: Matching 48 items
Brand 33: Matching 204 items
Bra

In [12]:
rows = []
for k, (best_match, best_score) in wf_item_to_food.items():
    rows.append((k, best_match['clean_desc'], best_match['fdc_id'], best_score))

wf_match_df = pd.DataFrame(rows, columns=['wf_name', 'usda_name', 'fdc_id', 'score'])
wf_match_df.to_sql("wf_match", conn, if_exists="replace", index=False)

rows = []
for k, (best_match, best_score) in wal_item_to_food.items():
    rows.append((k, best_match['clean_desc'], best_match['fdc_id'], best_score))

wal_match_df = pd.DataFrame(rows, columns=['wal_name', 'usda_name', 'fdc_id', 'score'])
wal_match_df.to_sql("wal_match", conn, if_exists="replace", index=False)

3103

In [16]:
conn.close()